# Model 4 — Pest Risk Forecaster (Synthetic, Kerala)

No good public dataset exists for district-level pest risk, so we generate a **synthetic dataset (10,000 rows)** using domain rules + noise.

Targets:
- `pest_risk_level`: Low/Medium/High
- `pest_name`

Model:
- Random Forest classifier (fast, robust)


In [ ]:
# Run this first in every notebook
!pip -q install scikit-learn pandas numpy matplotlib seaborn joblib

from google.colab import drive
drive.mount('/content/drive')

MODEL_SAVE_PATH = '/content/drive/MyDrive/krishi_mitra_models/'
DATA_PATH = '/content/drive/MyDrive/krishi_mitra_data/'

import os
from pathlib import Path
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)


## 1) Generate synthetic dataset

This imports the shared generator from `ml_models/data/generate_synthetic_data.py`.
If you are running only in Colab, you can copy that file into your Colab runtime or paste it into a cell.


In [ ]:
import pandas as pd
import numpy as np

# If you opened this notebook from the repo, this import works.
# If not, upload the repo folder or copy generate_synthetic_data.py into the runtime.
from ml_models.data.generate_synthetic_data import generate_pest_risk_dataset

df = generate_pest_risk_dataset(n_rows=10_000, seed=42)
df.head()

In [ ]:
df['pest_risk_level'].value_counts()

## 2) Train model (Random Forest)

We predict **risk level** and separately recommend a likely **pest_name**.
For simplicity, we model `pest_name` directly (multi-class). You can split into two models if desired.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

feature_cols = [
    'district','crop','month','avg_temperature','humidity_percent',
    'rainfall_last_7days_mm','wind_speed_kmh','previous_pest_incident','crop_growth_stage','cloudy'
]

X = df[feature_cols].copy()
y_risk = df['pest_risk_level'].astype(str)
y_pest = df['pest_name'].astype(str)

cat_cols = ['district','crop','crop_growth_stage']
num_cols = [c for c in feature_cols if c not in cat_cols]

pre = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', 'passthrough', num_cols),
])

X_train, X_test, y_train, y_test = train_test_split(X, y_risk, test_size=0.2, random_state=42, stratify=y_risk)

risk_model = Pipeline([
    ('pre', pre),
    ('clf', RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1, class_weight='balanced'))
])
risk_model.fit(X_train, y_train)
pred = risk_model.predict(X_test)
print(classification_report(y_test, pred))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, pred, labels=['Low','Medium','High'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Low','Medium','High'], yticklabels=['Low','Medium','High'])
plt.title('Risk level confusion matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

## 3) Save model

Saves to:
- local project path: `ml_models/4_pest_risk_forecaster/model/pest_model.pkl`
- Drive folder: `MODEL_SAVE_PATH/pest_risk_forecaster/`


In [ ]:
import joblib
from pathlib import Path

local_dir = Path('ml_models/4_pest_risk_forecaster/model')
local_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(risk_model, local_dir / 'pest_model.pkl')

drive_dir = Path(MODEL_SAVE_PATH) / 'pest_risk_forecaster'
drive_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(risk_model, drive_dir / 'pest_model.pkl')

print('Saved to local:', local_dir)
print('Saved to drive:', drive_dir)


## 4) `predict()` function

Returns a list of likely pest risks with probability and a practical advisory.


In [ ]:
PRACTICAL_ADVICE = {
    'Brown_Plant_Hopper': {
        'preventive': 'Avoid excess nitrogen; maintain proper water level; encourage natural predators.',
        'spray': 'If high: neem oil 3 ml/L or recommended insecticide per local extension advice.'
    },
    'Leaf_Blast': {
        'preventive': 'Ensure spacing and airflow; avoid late evening overhead watering.',
        'spray': 'If high: tricyclazole-based fungicide as per label/KAU guidance.'
    },
    'Aphids': {
        'preventive': 'Use yellow sticky traps; manage weeds; avoid water stress.',
        'spray': 'If high: neem oil/soap spray; chemical only if infestation severe.'
    },
    'Rhinoceros_Beetle': {
        'preventive': 'Destroy breeding sites (rotting logs); use pheromone traps.',
        'spray': 'If high: apply recommended granular/insecticidal measures to crown as advised.'
    },
    'Stem_Borer': {
        'preventive': 'Remove stubbles; synchronize planting; avoid excess nitrogen.',
        'spray': 'If high: follow local IPM schedule; use pheromone traps where available.'
    },
    'Phytophthora': {
        'preventive': 'Improve drainage; avoid collar wetness; prune for airflow.',
        'spray': 'If high: metalaxyl+mancozeb or copper-based sprays as recommended.'
    },
    'Pollu_Beetle': {
        'preventive': 'Maintain shade balance; avoid drought stress; sanitation.',
        'spray': 'If medium/high: follow pepper IPM guidance from local extension.'
    },
    'None': {
        'preventive': 'Continue weekly scouting; maintain field hygiene.',
        'spray': 'No spray recommended.'
    }
}

def predict(farmer_input: dict, top_k: int = 3):
    required = ['district','crop','month','temperature','humidity','rainfall_7days','crop_growth_stage']
    missing = [k for k in required if k not in farmer_input]
    if missing:
        raise ValueError(f"Missing keys: {missing}")

    row = {
        'district': farmer_input['district'],
        'crop': farmer_input['crop'],
        'month': int(farmer_input['month']),
        'avg_temperature': float(farmer_input['temperature']),
        'humidity_percent': float(farmer_input['humidity']),
        'rainfall_last_7days_mm': float(farmer_input['rainfall_7days']),
        'wind_speed_kmh': float(farmer_input.get('wind_speed', 9.0)),
        'previous_pest_incident': int(bool(farmer_input.get('previous_pest_incident', False))),
        'crop_growth_stage': farmer_input['crop_growth_stage'],
        'cloudy': int(bool(farmer_input.get('cloudy', False))),
    }

    X_one = pd.DataFrame([row])
    proba = risk_model.predict_proba(X_one)[0]
    classes = list(risk_model.named_steps['clf'].classes_)
    idx = np.argsort(proba)[::-1]

    # Heuristic pest_name recommendation based on inputs
    pest = 'None'
    if row['crop'] == 'Paddy' and row['humidity_percent'] > 80 and 25 <= row['avg_temperature'] <= 30:
        pest = 'Brown_Plant_Hopper'
    if row['crop'] == 'Paddy' and row['humidity_percent'] > 90 and 20 <= row['avg_temperature'] <= 25 and row['cloudy'] == 1:
        pest = 'Leaf_Blast'
    if row['avg_temperature'] > 30 and row['humidity_percent'] < 70 and row['rainfall_last_7days_mm'] < 20:
        pest = 'Aphids'
    if row['crop'] == 'Coconut':
        pest = 'Rhinoceros_Beetle'
    if row['crop'] == 'Paddy' and row['crop_growth_stage'] == 'Vegetative' and row['rainfall_last_7days_mm'] > 30:
        pest = 'Stem_Borer'
    if row['crop'] == 'Black_Pepper' and row['humidity_percent'] > 88 and row['rainfall_last_7days_mm'] > 60:
        pest = 'Phytophthora'

    advice = PRACTICAL_ADVICE.get(pest, PRACTICAL_ADVICE['None'])

    out = []
    for j in idx[:top_k]:
        out.append({
            'pest_name': pest,
            'risk_level': classes[j].lower(),
            'probability': float(proba[j]),
            'peak_window': 'next 3-5 days',
            'preventive_action': advice['preventive'],
            'spray_recommendation': advice['spray'],
        })
    return out

test_farmer = {
    'district': 'Palakkad',
    'crop': 'Paddy',
    'month': 7,
    'temperature': 28,
    'humidity': 88,
    'rainfall_7days': 90,
    'crop_growth_stage': 'Vegetative',
    'previous_pest_incident': True,
    'cloudy': True,
}

predict(test_farmer)